# DCGAN on Google Colab (Free GPU)

This notebook trains the DCGAN on Fashion-MNIST using Colab's free T4 GPU.

## Quick Start
1. **Runtime → Change runtime type → GPU (T4)**
2. Run all cells (Ctrl+F9)
3. Check `samples/` for generated images
4. Download checkpoints/results before session ends

In [ ]:
# Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Clone or upload project
# Option A: From GitHub (RECOMMENDED - fastest)
!git clone https://github.com/anand-esc/dcgan-mnist.git
%cd dcgan-mnist

# Option B: Upload from local (if not using GitHub)
# from google.colab import files
# print('Upload dcgan_ibm_project_colab/dcgan_ibm_project as .zip')
# uploaded = files.upload()
# !unzip -q *.zip
# %cd dcgan_ibm_project_colab/dcgan_ibm_project

import os
print(f'Working in: {os.getcwd()}')
print('Files:', os.listdir('.'))

In [ ]:
# Install dependencies
!pip install -r requirements.txt -q

# Verify
import torch, torchvision, numpy, matplotlib, tqdm, tensorboard, PIL
print('All packages imported successfully')

In [ ]:
# Quick config check
import json
with open('configs/config.json') as f:
    config = json.load(f)
print(json.dumps(config, indent=2))

In [ ]:
# Train (50 epochs ~ 5-10 min on T4)
!python main.py train

# If interrupted, resume:
# !python main.py train --resume checkpoints/checkpoint_epoch_XXXX.pth

In [ ]:
# View generated samples
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

sample_files = sorted([f for f in os.listdir('samples') if f.endswith('.png')])
print(f'Sample files: {sample_files}')

# Show latest
if sample_files:
    latest = sample_files[-1]
    img = mpimg.imread(os.path.join('samples', latest))
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Latest: {latest}')
    plt.show()

In [ ]:
# View loss curves
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import os

loss_path = 'samples/loss_curves.png'
if os.path.exists(loss_path):
    img = mpimg.imread(loss_path)
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print('Loss curves not generated yet (training may not have completed)')

In [ ]:
# Generate more samples / interpolations
!python main.py evaluate --checkpoint checkpoints/checkpoint_final.pth --n_samples 64
!python main.py evaluate --checkpoint checkpoints/checkpoint_final.pth --interpolation

In [ ]:
# Download results (run before session ends!)
from google.colab import files
import zipfile, os

# Create zip of checkpoints and samples
with zipfile.ZipFile('dcgan_results.zip', 'w') as zf:
    for root, dirs, filenames in os.walk('checkpoints'):
        for f in filenames:
            zf.write(os.path.join(root, f))
    for root, dirs, filenames in os.walk('samples'):
        for f in filenames:
            zf.write(os.path.join(root, f))
    for root, dirs, filenames in os.walk('logs'):
        for f in filenames:
            zf.write(os.path.join(root, f))

files.download('dcgan_results.zip')
print('Downloaded dcgan_results.zip')

## TensorBoard (Optional)

Run in a separate cell to view live metrics:

In [ ]:
# TensorBoard
%load_ext tensorboard
%tensorboard --logdir logs --port 6006

## Notes

- **Session timeout:** Colab disconnects after ~90 min inactivity. Download results!
- **Free tier limits:** 12-hr max session, may get K80 instead of T4
- **Pro tip:** Mount Google Drive to persist checkpoints:
  ```python
  from google.colab import drive
  drive.mount('/content/drive')
  # Then symlink checkpoints to drive
  ```
- **To push to GitHub from Colab (if you cloned):**
  ```bash
  !git config --global user.email "you@example.com"
  !git config --global user.name "Your Name"
  !git add . && git commit -m "Colab trained"
  !git push
  ```